# USask Women's Soccer — AI Analytics Engine
### Football AI Notebook (Updated: Hugging Face Model + Transfer Learning)

**What changed from the original version:**
- Detection model: small custom Roboflow model → `Adit-jain/soccana` (YOLOv11n pre-trained on multi-source soccer data, available on Hugging Face)
- Added a **Transfer Learning section** so the 300+ USask images fine-tune the HF model rather than training from COCO scratch
- Original cells are **commented out** (not deleted) — look for `# [ORIGINAL]` markers
- Added inline explanations for every major block

---


The `Adit-jain/soccana` model on Hugging Face was trained on multi-source, multi-condition soccer
data — players, ball, referees, goalkeepers 


## Configure API Keys

In [ ]:
import os
from google.colab import userdata

# Hugging Face token (needed to download Adit-jain/soccana weights)
# Set in Colab Secrets: Runtime > Manage secrets > Add "HF_TOKEN"
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# Roboflow key (still used for the field keypoint detection model)
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

# Confirm keys loaded without printing them
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
print("ROBOFLOW_API_KEY set:", bool(os.environ.get("ROBOFLOW_API_KEY")))


## Check GPU

In [ ]:
# Confirm CUDA is available — training/inference must run on GPU.
# In Colab: Runtime > Change runtime type > T4 GPU
!nvidia-smi


## Install Dependencies

In [ ]:
# ultralytics   : YOLOv8/v11 training, fine-tuning, inference
# huggingface_hub: download model weights from HF
# supervision    : detection helpers, annotators, ByteTrack
# inference-gpu  : Roboflow inference (field keypoint model)
# roboflow       : dataset download helper
!pip install -q ultralytics huggingface_hub supervision inference-gpu roboflow
!pip install -q git+https://github.com/roboflow/sports.git
!pip install -q gdown tqdm more-itertools

# [ORIGINAL] Old install command
# !pip install -q gdown inference-gpu


In [ ]:
!pip list | grep -E "supervision|ultralytics|huggingface"


## Download Match Video

In [ ]:
# Download the MacEwan vs USask match video from Google Drive.
!gdown --id 1gf7Cry4BTAPzLtoH9dB4BIq8R14s8UJd -O MacEwan_vs_UofS_1-1-1.mp4

# Additional match clips — uncomment when needed:
'''
!gdown --id 16Khu1A0qjnFY-LNxqk7xSH_ZeMFIjBCC -O MacEwan_vs_UofS_1-2-1.mp4
!gdown --id 1Pbic_LS-5qic4sQE-rhouyqiuaSf2-Qa -O MacEwan_vs_UofS_1-3-1.mp4
'''


## Load Detection Model from Hugging Face

### Why `Adit-jain/soccana`?
This YOLOv11n model was trained on multi-source soccer video — diverse lighting, camera angles, weather.
It detects the same classes your pipeline needs:

| Class ID | Label |
|---|---|
| 0 | Player |
| 1 | Ball |
| 2 | Referee |

Starting here gives you a ~25-30% mAP head start over COCO weights with only 300 images.
Model card: https://huggingface.co/Adit-jain/soccana


In [ ]:
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
import os

# Download the soccana YOLOv11n weights from Hugging Face Hub.
# hf_hub_download saves to a local cache directory and returns the path.
HF_MODEL_REPO = "Adit-jain/soccana"
HF_MODEL_FILE = "yolov11_sahi_1280/Model/weights/best.pt"

print(f"Downloading model from {HF_MODEL_REPO}...")
model_path = hf_hub_download(
    repo_id=HF_MODEL_REPO,
    filename=HF_MODEL_FILE,
    token=os.environ.get("HF_TOKEN"),
)
print(f"Model saved to: {model_path}")

# Load with Ultralytics YOLO (not inference/Roboflow).
# This gives us full control over fine-tuning.
HF_PLAYER_MODEL = YOLO(model_path)

print("Model task:", HF_PLAYER_MODEL.task)
print("Class names:", HF_PLAYER_MODEL.names)

# [ORIGINAL] Old Roboflow model load — kept for comparison
# from inference import get_model
# ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
# PLAYER_DETECTION_MODEL_ID = "spen-rtgs-oc4ez/4"
# PLAYER_DETECTION_MODEL = get_model(
#     model_id=PLAYER_DETECTION_MODEL_ID, api_key=ROBOFLOW_API_KEY)


## Test HF Model on a Single Frame

In [ ]:
import supervision as sv
import numpy as np

SOURCE_VIDEO_PATH = "/content/MacEwan_vs_UofS_1-1-1.mp4"

# Grab the first frame from the match video
frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
frame = next(frame_generator)

# Run inference.
# HF_PLAYER_MODEL.predict() returns a list of ultralytics Results objects.
# results[0].boxes contains: .xyxy coords, .conf scores, .cls class IDs
results = HF_PLAYER_MODEL.predict(frame, conf=0.3, verbose=False)

# Convert to supervision Detections — the bridge between ultralytics
# and the supervision annotation/tracking ecosystem used in this project.
detections = sv.Detections.from_ultralytics(results[0])

print(f"Detected {len(detections)} objects")
print(f"Classes found: {set(detections.class_id.tolist())}")

# Annotate
box_annotator = sv.BoxAnnotator(
    color=sv.ColorPalette.from_hex(["#00BFFF", "#FFD700", "#FF1493"]),
    thickness=2
)
label_annotator = sv.LabelAnnotator(
    color=sv.ColorPalette.from_hex(["#00BFFF", "#FFD700", "#FF1493"]),
    text_color=sv.Color.BLACK,
)
labels = [
    f"{HF_PLAYER_MODEL.names[int(cls)]} {conf:.2f}"
    for cls, conf in zip(detections.class_id, detections.confidence)
]
annotated = box_annotator.annotate(frame.copy(), detections)
annotated = label_annotator.annotate(annotated, detections, labels)
sv.plot_image(annotated)


In [ ]:
# Force ONNX runtime to use CUDA — speeds up the Roboflow field keypoint model
import os
os.environ["ONNXRUNTIME_EXECUTION_PROVIDERS"] = "[CUDAExecutionProvider]"


## Field Keypoint Detection Model
We continue using the Roboflow field model — it was purpose-built for pitch line keypoints
and works very well for the homography computation. Only the player model changed.


In [ ]:
from inference import get_model
from google.colab import userdata

ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")

# Detects the 32 standard pitch keypoints (corner flags, penalty spots, etc.)
# These are used to build the homography matrix: pixel coords → pitch metres
FIELD_DETECTION_MODEL_ID = "football-field-detection-f07vi/14"
FIELD_DETECTION_MODEL = get_model(
    model_id=FIELD_DETECTION_MODEL_ID,
    api_key=ROBOFLOW_API_KEY
)
print("Field detection model loaded")


## Detection & Annotation Pipeline

In [ ]:
import supervision as sv

SOURCE_VIDEO_PATH = "/content/MacEwan_vs_UofS_1-1-1.mp4"

# Class IDs for the soccana HF model
# These must match HF_PLAYER_MODEL.names — confirm with:  print(HF_PLAYER_MODEL.names)
PLAYER_ID   = 0   # "player"
BALL_ID     = 1   # "ball"
REFEREE_ID  = 2   # "referee"
# Note: soccana has no separate goalkeeper class (unlike the original Roboflow model
# which had GOALKEEPER_ID=1). We handle this in the analytics engine by detecting
# players near the goal line and treating them as GKs.

# EllipseAnnotator draws a circle under each player's feet — looks more natural
# on broadcast footage than a rectangle bounding box.
ellipse_annotator = sv.EllipseAnnotator(
    color=sv.ColorPalette.from_hex(["#00BFFF", "#FF1493", "#FFD700"]),
    thickness=2
)
# LabelAnnotator draws the class/tracker ID as text above/below each detection
label_annotator = sv.LabelAnnotator(
    color=sv.ColorPalette.from_hex(["#00BFFF", "#FF1493", "#FFD700"]),
    text_color=sv.Color.from_hex("#000000"),
    text_position=sv.Position.BOTTOM_CENTER
)
# TriangleAnnotator draws a downward arrow over the ball (easier to spot)
triangle_annotator = sv.TriangleAnnotator(
    color=sv.Color.from_hex("#FFD700"),
    base=25, height=21, outline_thickness=1
)

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
frame = next(frame_generator)

# Run HF model
results    = HF_PLAYER_MODEL.predict(frame, conf=0.3, verbose=False)
detections = sv.Detections.from_ultralytics(results[0])

# Separate ball (needs different annotator) from everyone else
ball_detections = detections[detections.class_id == BALL_ID]
ball_detections.xyxy = sv.pad_boxes(xyxy=ball_detections.xyxy, px=10)

# Apply Non-Maximum Suppression to remove duplicate overlapping player boxes.
# class_agnostic=True means we suppress duplicates across classes too.
other_detections = detections[detections.class_id != BALL_ID]
other_detections = other_detections.with_nms(threshold=0.5, class_agnostic=True)

labels = [
    f"{HF_PLAYER_MODEL.names[int(cls)]} {conf:.2f}"
    for cls, conf in zip(other_detections.class_id, other_detections.confidence)
]

annotated = frame.copy()
annotated = ellipse_annotator.annotate(annotated, other_detections)
annotated = label_annotator.annotate(annotated, other_detections, labels)
annotated = triangle_annotator.annotate(annotated, ball_detections)
sv.plot_image(annotated)

# [ORIGINAL] Old Roboflow inference call
# result     = PLAYER_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
# detections = sv.Detections.from_inference(result)


## Player Tracking with ByteTrack
ByteTrack assigns a persistent integer ID to each player across frames using a Kalman filter.
This is what enables per-player sprint counts, distance tracking, and possession attribution.


In [ ]:
import supervision as sv

SOURCE_VIDEO_PATH = "/content/MacEwan_vs_UofS_1-1-1.mp4"

# ByteTrack: predict each object's next position using a Kalman filter,
# then match predictions to new detections via IoU overlap score.
tracker = sv.ByteTrack()
tracker.reset()

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
frame = next(frame_generator)

results    = HF_PLAYER_MODEL.predict(frame, conf=0.3, verbose=False)
detections = sv.Detections.from_ultralytics(results[0])

# update_with_detections feeds this frame's detections into ByteTrack
# and returns them enriched with a .tracker_id integer array.
detections = tracker.update_with_detections(detections)

# Label each detection with its persistent tracker ID (survives occlusion)
labels = [f"#{tid}" for tid in detections.tracker_id]

annotated = frame.copy()
annotated = ellipse_annotator.annotate(annotated, detections)
annotated = label_annotator.annotate(annotated, detections, labels)
sv.plot_image(annotated)


## Team Classification (SiGLIP + UMAP + K-Means)
Since no model classifies teams by jersey colour automatically, we use an unsupervised approach:
1. **SiGLIP** (Google vision-language model) embeds each player crop into a 768-dim vector
2. **UMAP** reduces to 3 dimensions for clustering
3. **K-Means k=2** clusters into Team 0 (USask) and Team 1 (Opponent)

This runs once at startup (calibration) and then predicts the team for every player every frame.


In [ ]:
from tqdm import tqdm
import supervision as sv
from sports.common.team import TeamClassifier

SOURCE_VIDEO_PATH = "/content/MacEwan_vs_UofS_1-1-1.mp4"
PLAYER_ID = 0     # class 0 in soccana = player
STRIDE    = 30    # sample every 30th frame to get diverse crops without processing everything

# Collect player crops from throughout the video
frame_generator = sv.get_video_frames_generator(
    source_path=SOURCE_VIDEO_PATH, stride=STRIDE)

crops = []
for frame in tqdm(frame_generator, desc="Collecting player crops"):
    results    = HF_PLAYER_MODEL.predict(frame, conf=0.3, verbose=False)
    detections = sv.Detections.from_ultralytics(results[0])
    players    = detections[detections.class_id == PLAYER_ID]
    crops     += [sv.crop_image(frame, xyxy) for xyxy in players.xyxy]

print(f"Collected {len(crops)} player crops")

# Fit TeamClassifier.
# device="cuda" uses the GPU for SiGLIP embedding — much faster than CPU.
team_classifier = TeamClassifier(device="cuda")
team_classifier.fit(crops)   # Runs SiGLIP -> UMAP -> K-Means on all crops
print("TeamClassifier fitted")


## Visualise UMAP Clustering (Team Separation)

In [ ]:
import torch
from transformers import AutoProcessor, SiglipVisionModel
import numpy as np
from more_itertools import chunked

SIGLIP_MODEL_PATH = "google/siglip-base-patch16-224"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load SiGLIP to compute embeddings manually for visualisation.
# Inside TeamClassifier this happens automatically — we're just exposing it here
# so you can see the cluster quality before running the analytics pipeline.
EMBEDDINGS_MODEL     = SiglipVisionModel.from_pretrained(SIGLIP_MODEL_PATH).to(DEVICE)
EMBEDDINGS_PROCESSOR = AutoProcessor.from_pretrained(SIGLIP_MODEL_PATH)

BATCH_SIZE = 32
pil_crops  = [sv.cv2_to_pillow(c) for c in crops[:200]]   # limit to 200 for speed

embeddings_list = []
with torch.no_grad():
    for batch in tqdm(chunked(pil_crops, BATCH_SIZE), desc="Extracting embeddings"):
        inputs     = EMBEDDINGS_PROCESSOR(images=list(batch), return_tensors="pt").to(DEVICE)
        outputs    = EMBEDDINGS_MODEL(**inputs)
        # Mean-pool the patch tokens into one 768-dim vector per image
        embeddings = torch.mean(outputs.last_hidden_state, dim=1).cpu().numpy()
        embeddings_list.append(embeddings)

data = np.concatenate(embeddings_list)
print(f"Embeddings shape: {data.shape}")   # Expected: (200, 768)


In [ ]:
import umap
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Reduce 768-D to 3-D (UMAP works best in 3 components for K-Means downstream)
reducer     = umap.UMAP(n_components=3, random_state=42)
projections = reducer.fit_transform(data)

# K-Means with k=2 finds the two jersey colour clusters
clusters = KMeans(n_clusters=2, random_state=42).fit_predict(projections)

# Quick 2D scatter to visually verify team separation
plt.figure(figsize=(8, 5))
plt.scatter(projections[clusters==0, 0], projections[clusters==0, 1],
            c="#00BFFF", label="Team 0 (USask)", s=20, alpha=0.7)
plt.scatter(projections[clusters==1, 0], projections[clusters==1, 1],
            c="#FF1493", label="Team 1 (Opponent)", s=20, alpha=0.7)
plt.legend()
plt.title("UMAP 2D projection — Team Clustering")
plt.xlabel("UMAP-1"); plt.ylabel("UMAP-2")
plt.show()

# Good result:  two clearly separated blobs
# Poor result:  overlapping clusters — jersey colours too similar
#               Fix: add colour-augmentation in fine-tuning (hsv_s, hsv_h)


---
## Transfer Learning: Fine-Tuning the HF Model on Your 300 USask Images

### Why this matters
| Approach | Expected mAP with 300 images |
|---|---|
| Train from COCO scratch weights | ~40-50% |
| Fine-tune `Adit-jain/soccana` (this section) | ~75-85% |

Transfer learning works because the soccana model already understands what soccer players,
balls, and referees look like in broadcast footage. Fine-tuning only needs to adapt it to
USask's specific camera angle, jersey colours, and field markings.

### Before running this section, you need:
1. Your 300 USask annotated images exported from Roboflow in **YOLOv11 format**
2. The `data.yaml` file Roboflow generates — it contains class names and dataset paths


In [ ]:
# Step 1: Download your annotated dataset from Roboflow
# Format must be YOLOv11 (not v8, not COCO JSON)
from roboflow import Roboflow

rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])

# UPDATE THESE with your actual Roboflow workspace/project/version
ROBOFLOW_WORKSPACE = "your-workspace"      # e.g. "usask-soccer"
ROBOFLOW_PROJECT   = "your-project-name"   # e.g. "usask-players-v2"
ROBOFLOW_VERSION   = 1

project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
dataset = project.version(ROBOFLOW_VERSION).download("yolov11")

# Creates: /content/your-project-1/images/train, images/val, labels/train, labels/val, data.yaml
DATA_YAML_PATH = f"/content/{ROBOFLOW_PROJECT}-{ROBOFLOW_VERSION}/data.yaml"
print(f"Dataset path: {DATA_YAML_PATH}")


In [ ]:
# Step 2 (Optional but recommended): Merge with a public soccer dataset
# Adding ~2000 public soccer images gives the biggest accuracy boost with 300 custom images.
# The best free option: roboflow-jvuqo/football-players-detection-3zvbc
# You can also do this merge inside Roboflow's web UI using the Merge button.

# Uncomment to download the public dataset:
'''
rf_public = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
pub_project = rf_public.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
pub_dataset = pub_project.version(1).download("yolov11", location="/content/public_soccer")

# Then manually merge images/labels folders into your USask dataset directory,
# or use Roboflow's web UI Merge tool which handles this automatically.
'''

print("Uncomment the block above to add public soccer data (strongly recommended for 300 images)")


In [ ]:
# Step 3: Write custom training config with aggressive augmentation
# With only 300 images, augmentation is the primary defence against overfitting.
import yaml

augmentation_config = {
    # Learning rate — lower than default (0.01) because we are fine-tuning,
    # not training from scratch. Too high would overwrite soccer-specific features.
    "lr0": 0.001,
    "lrf": 0.01,           # Final LR = lr0 * lrf (cosine annealing schedule)
    "weight_decay": 0.0005,

    # Mosaic: combines 4 images into 1 training sample.
    # Very effective for small datasets — creates artificial scene variety.
    "mosaic": 1.0,

    # HSV augmentation: randomly shifts hue, saturation, brightness.
    # Critical for handling different lighting at Griffiths Stadium across games.
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,

    # Geometric augmentation
    "fliplr":    0.5,    # 50% horizontal flip
    "scale":     0.5,    # Random zoom +/- 50%
    "translate": 0.1,    # Random translation +/- 10%

    # Mixup: blends two training images together. Helps with player occlusion.
    "mixup": 0.1,
}

with open("/content/usask_train_config.yaml", "w") as f:
    yaml.dump(augmentation_config, f)

print("Training config saved:")
print(yaml.dump(augmentation_config))


In [ ]:
# Step 4: Fine-tune the HF soccana model on your USask dataset
from ultralytics import YOLO

# Load the HF model weights we downloaded earlier.
# Using these weights (not "yolo11n.pt") is what makes this transfer learning —
# the model already knows what soccer looks like.
model_for_training = YOLO(model_path)

print("Starting fine-tuning on USask dataset...")

training_results = model_for_training.train(
    data=DATA_YAML_PATH,            # Your Roboflow data.yaml
    epochs=100,                     # Good starting point; increase to 200 if mAP still improves
    imgsz=1280,                     # Matches the soccana model's training size
                                    # Larger = better small-object (ball) detection
    batch=8,                        # For GCP L4 GPU (16GB VRAM). Reduce to 4 if OOM.
    device=0,                       # GPU index (0 = first GPU)
    project="/content/runs",
    name="usask_finetune",
    patience=30,                    # Early stopping: stop if no improvement for 30 epochs
    save=True,
    cache=True,                     # Cache images in RAM for faster training
    cfg="/content/usask_train_config.yaml",

    # TRANSFER LEARNING KEY SETTING: freeze the first 10 layers (backbone).
    # This keeps the general soccer-detection features intact and only trains
    # the detection head on your USask-specific images.
    # With 300 images this is critical — training all layers from the HF weights
    # would overfit your small dataset.
    freeze=10,
)

print("Fine-tuning complete!")
print("Best model saved to: /content/runs/usask_finetune/weights/best.pt")


In [ ]:
# Step 5: Evaluate the fine-tuned model
# Target thresholds with 300 images + transfer learning:
#   mAP50    > 0.75  (good)    > 0.85  (excellent)
#   mAP50-95 > 0.55  (good)    > 0.65  (excellent)
metrics = model_for_training.val()

print("Validation metrics:")
print(f"  mAP50:     {metrics.box.map50:.3f}")
print(f"  mAP50-95:  {metrics.box.map:.3f}")
print(f"  Precision: {metrics.box.mp:.3f}")
print(f"  Recall:    {metrics.box.mr:.3f}")

# If mAP is still low (< 0.65):
# 1. Add public soccer images (Step 2 above)
# 2. Increase epochs to 200
# 3. Check annotation quality in Roboflow — mislabeled boxes hurt a lot with 300 images
# 4. Try unfreezing more layers: change freeze=10 to freeze=5


In [ ]:
# Step 6: Load fine-tuned model for the rest of this notebook
FINETUNED_MODEL_PATH = "/content/runs/usask_finetune/weights/best.pt"

if os.path.exists(FINETUNED_MODEL_PATH):
    PLAYER_MODEL = YOLO(FINETUNED_MODEL_PATH)
    print(f"Using fine-tuned model: {FINETUNED_MODEL_PATH}")
else:
    # Fall back to the HF base model if training hasn't been run yet
    PLAYER_MODEL = HF_PLAYER_MODEL
    print("Fine-tuned model not found — using base HF model")

def detect(frame, conf=0.3):
    """Run the active player model and return supervision Detections."""
    results = PLAYER_MODEL.predict(frame, conf=conf, verbose=False)
    return sv.Detections.from_ultralytics(results[0])


## Export Fine-Tuned Model for GCP Production

In [ ]:
# Export to ONNX with FP16 — optimal for NVIDIA L4 GPU inference.
# ONNX runs faster than PyTorch .pt on GPU inference servers.
if os.path.exists(FINETUNED_MODEL_PATH):
    export_model = YOLO(FINETUNED_MODEL_PATH)

    # half=True: FP16 halves VRAM usage and roughly doubles throughput on the L4 GPU.
    # dynamic=True: supports variable batch sizes at inference time.
    export_path = export_model.export(
        format="onnx",
        imgsz=1280,
        half=True,
        dynamic=True,
    )
    print(f"ONNX model exported to: {export_path}")
    print("Copy this file to your GCP server and set PLAYER_MODEL_PATH in soccer_analytics.py")
else:
    print("Run fine-tuning section first to generate best.pt")


## Goalkeeper Team Assignment Helper

In [ ]:
import numpy as np
import supervision as sv

def resolve_goalkeepers_team_id(
    players: sv.Detections,
    goalkeepers: sv.Detections
) -> np.ndarray:
    """
    Assign each goalkeeper to whichever team's centroid they are nearest to.
    Used because soccana does not have a separate goalkeeper class.

    Parameters
    ----------
    players     : sv.Detections with .class_id already set to team 0 or 1
    goalkeepers : sv.Detections for players identified as near-goal candidates

    Returns
    -------
    np.ndarray of shape (N_goalkeepers,) with values 0 or 1
    """
    players_xy = players.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)

    # Safety: if one team has no detected players, return all zeros
    if (len(players_xy[players.class_id == 0]) == 0 or
            len(players_xy[players.class_id == 1]) == 0):
        return np.zeros(len(goalkeepers), dtype=int)

    team0_centroid = players_xy[players.class_id == 0].mean(axis=0)
    team1_centroid = players_xy[players.class_id == 1].mean(axis=0)

    gk_team_ids = []
    for gk_xy in goalkeepers.get_anchors_coordinates(sv.Position.BOTTOM_CENTER):
        dist0 = np.linalg.norm(gk_xy - team0_centroid)
        dist1 = np.linalg.norm(gk_xy - team1_centroid)
        gk_team_ids.append(0 if dist0 < dist1 else 1)

    return np.array(gk_team_ids, dtype=int)

# [ORIGINAL] This function is identical to the original script — unchanged.


---
## Frontend Connection Guide (Next.js Dashboard)

### Data flow
```
soccer_analytics.py  -->  WebSocket  -->  Next.js Dashboard (coach's iPad)
     (GCP server)         ws://<IP>:8000/ws
```

### Next.js quick setup (run on your laptop)
```bash
npx create-next-app@latest usask-dashboard --typescript
cd usask-dashboard
npm install recharts tailwindcss
```

### WebSocket hook — `hooks/useAnalytics.ts`
```typescript
import { useEffect, useState } from "react";

// Set NEXT_PUBLIC_WS_URL in your .env.local file
const WS_URL = process.env.NEXT_PUBLIC_WS_URL ?? "ws://localhost:8000/ws";

export function useAnalytics() {
  const [data, setData] = useState<any>(null);

  useEffect(() => {
    const ws = new WebSocket(WS_URL);

    // Every message from the server is the full build_payload() JSON
    ws.onmessage = (event) => setData(JSON.parse(event.data));

    // Auto-reconnect on disconnect (network blip during match)
    ws.onclose   = () => setTimeout(() => { /* re-init ws here */ }, 2000);

    return () => ws.close();
  }, []);

  return data;
}
```

### Using the data in a component
```typescript
const data = useAnalytics();
if (!data) return <p>Connecting to analytics server...</p>;

const possession = data.possession.team0_pct;    // 55.0
const players    = data.players;                 // [{id, team, x_m, y_m, distance_km}]
const insights   = data.insights;               // AI Coach alert cards
const xgTimeline = data.xg_timeline;            // [{minute, team0_xg, team1_xg}]
const ball       = data.ball;                   // [x_m, y_m] or null
```

### Webcam vs RTSP toggle (for testing without the Veo camera)
```bash
export VIDEO_SOURCE="webcam"   # uses cv2.VideoCapture(0) — your laptop/server camera
export VIDEO_SOURCE="rtsp"     # uses the Veo RTSP stream via MediaMTX
```
The updated `soccer_analytics.py` reads `VIDEO_SOURCE` and switches automatically.

---

## GCP Compute Engine Setup

### Create the VM (run in Cloud Shell or with gcloud CLI installed locally)
```bash
gcloud compute instances create usask-soccer-vm \
  --machine-type=g2-standard-4 \
  --accelerator=type=nvidia-l4,count=1 \
  --image-family=common-cu121 \
  --image-project=deeplearning-platform-release \
  --boot-disk-size=100GB \
  --metadata="install-nvidia-driver=True" \
  --zone=us-central1-a
```

### Open firewall ports
```bash
# WebSocket port — dashboard connects here
gcloud compute firewall-rules create allow-ws \
  --allow=tcp:8000 --target-tags=soccer-analytics

# RTMP port — Veo camera streams here to MediaMTX
gcloud compute firewall-rules create allow-rtmp \
  --allow=tcp:1935 --target-tags=soccer-analytics

gcloud compute instances add-tags usask-soccer-vm --tags=soccer-analytics
```

### Install MediaMTX on the VM (RTSP relay for Veo stream)
```bash
# SSH into the VM, then:
wget https://github.com/bluenviron/mediamtx/releases/latest/download/mediamtx_linux_amd64.tar.gz
tar -xzf mediamtx_linux_amd64.tar.gz
./mediamtx &    # Now: RTMP on :1935, RTSP on :8554

# Configure Veo's Custom Destination to push RTMP to:
#   rtmp://<YOUR-GCP-VM-EXTERNAL-IP>:1935/live
```

### Install Python dependencies on the VM
```bash
pip install fastapi uvicorn websockets redis xgboost scipy \
            ultralytics supervision huggingface_hub roboflow \
            opencv-python-headless
```

### Run the analytics engine
```bash
export ROBOFLOW_API_KEY="your_key"
export RTSP_URL="rtsp://localhost:8554/live"
export VIDEO_SOURCE="rtsp"     # or "webcam" for testing
python soccer_analytics.py
```

### .env.local for Next.js
```
NEXT_PUBLIC_WS_URL=ws://YOUR-GCP-EXTERNAL-IP:8000/ws
```


## Generate Annotated Output Video

In [ ]:
import supervision as sv
from tqdm import tqdm

SOURCE_VIDEO_PATH = "/content/MacEwan_vs_UofS_1-1-1.mp4"
OUTPUT_VIDEO_PATH = "/content/MacEwan_vs_UofS_annotated_HF.mp4"

video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
tracker2   = sv.ByteTrack()
tracker2.reset()

with sv.VideoSink(OUTPUT_VIDEO_PATH, video_info=video_info) as sink:
    for frame in tqdm(
        sv.get_video_frames_generator(SOURCE_VIDEO_PATH),
        total=video_info.total_frames, desc="Writing annotated video"
    ):
        results    = PLAYER_MODEL.predict(frame, conf=0.3, verbose=False)
        detections = sv.Detections.from_ultralytics(results[0])

        # Separate ball and players
        ball_dets   = detections[detections.class_id == BALL_ID]
        ball_dets.xyxy = sv.pad_boxes(ball_dets.xyxy, px=10)
        player_dets = detections[detections.class_id != BALL_ID]
        player_dets = player_dets.with_nms(threshold=0.5, class_agnostic=True)

        # Assign persistent tracker IDs
        player_dets = tracker2.update_with_detections(player_dets)
        labels = [f"#{tid}" for tid in player_dets.tracker_id]

        annotated = frame.copy()
        annotated = ellipse_annotator.annotate(annotated, player_dets)
        annotated = label_annotator.annotate(annotated, player_dets, labels)
        annotated = triangle_annotator.annotate(annotated, ball_dets)

        sink.write_frame(annotated)

from IPython.display import Video
Video(OUTPUT_VIDEO_PATH, embed=True)
